# Лабораторная работа 6. Машинный перевод

#### Работу выполнил: {впишите вашу фамилию} {впишите ваше имя}, студент группы ФИб-4

В данной лабораторной работе познакомимся с моделями типа seq-to-seq, который позволяют по текстовому входу получать текст. В основе данных моделей лежит энкодер-декодерная архитектура, где часть `encoder` отвечает за представление/кодирование входных данных в некоторое скрытое пространство, а часть `decoder`выполняет разворачивание/раскодирование в результат.

Примером такой модели является модель `T5`. Познакомьтесь со <a href="https://habr.com/ru/articles/762140/">статьей</a>.

**Важно**: в меню `colab` выберите среду выполнения `GPU` (`Runtime` -> `Change runtime type`)

Установите следующие пакеты/библиоткеки, которые понадобятся для дальнейшей работы.

In [ ]:
%pip install datasets

In [1]:
%pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.3 MB/s eta 0:00:00


In [2]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and

Подключите следующие пакеты

In [4]:
from transformers import AutoTokenizer
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from transformers import DataCollatorForSeq2Seq
import numpy as np
import torch
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split
import evaluate

Используя библиотеку `transformers`, можно удобно загружать предобученные модели и их токенайзеры, которые отвечают за работу со словарём (в предыдущей лабораторной работе для этого добавлялись отдельные свои функции)

Загрузите модель `https://huggingface.co/ai-forever/ruT5-base`, используя библиотеку `transformers`.

In [ ]:
model_path = "ai-forever/ruT5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, use_cache=False).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_path)

Входной текст для модели будет содержать `формулировку задания` и `текст на русском языке`, который нужно перевести на английский язык.

Подготовим переменные и функцию предварительной обработки, которые понадобятся для подготовки текстов из датасета для обучения.

In [ ]:
english_lang = "en"
russian_lang = "ru"
source_lang = english_lang
target_lang = russian_lang
prefix = f"translate from english to russian: "

In [ ]:
def preprocess_function(examples):
  inputs = [prefix + example[source_lang] for example in examples["translation"]]
  targets = [example[target_lang] for example in examples["translation"]]
  model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
  return model_inputs

Подготовим функцию для генерации. У загруженной модели в класс `AutoModelForSeq2SeqLM` есть метод `generate`, который позволяет получать от модели сгенерированный текст по входному тексту. Перед тем как передать входной текст в модель его нужно преобразовать в массив чисел (сначала получить токены, а затем сформировать индексы токенов в словаре) - это выполняет `tokenizer`. Результатов `прямого прохода` модель выдаст массив индекстов, которые нужно обратно собрать в текст - это можно сдедать через метод `decode` у `tokenizer` .

In [ ]:
def generate(text):
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        hypotheses = model.generate(**inputs, max_new_tokens=100, num_beams=5, early_stopping=True)
    return tokenizer.decode(hypotheses[0], skip_special_tokens=True)

Модель предобучалась на задаче заполнения пропусков в тексте. Проверим как она подбирает слова в тексте на место тех, которые закрыты специальными токенами `extra_id_{i}`

In [ ]:
generate("Кошка сидит <extra_id_0> и ест <extra_id_1> колбасу.")

Создадим `collator`, который будет передавать данные в процесе обучения модели.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_path)

Загрузим датасет `https://huggingface.co/datasets/Helsinki-NLP/opus-100` и подготовим его для обучения.

In [ ]:
raw_dataset = load_dataset("Helsinki-NLP/opus-100", "en-ru", split="train[:2500]").shuffle().train_test_split(test_size=0.2)
tokenized_dataset = raw_dataset.map(preprocess_function, batched=True)
tokenized_dataset

Посмотрим пример из датасета.

In [ ]:
raw_dataset["train"][0]

{'id': '16234',
 'translation': {'en': 'The boulevard and children.', 'ru': 'Бульвар и дети.'}}

Для оценивания качества перевода будем использовать метрику BLEU.

In [ ]:
metric = evaluate.load("sacrebleu")

/usr/local/lib/python3.10/dist-packages/datasets/load.py:756: FutureWarning: The repository for sacrebleu contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/sacrebleu/sacrebleu.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [ ]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds[preds == -100] = tokenizer.pad_token_id
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [ ]:
training_args = Seq2SeqTrainingArguments(
    run_name="ruT5_base_mt",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    learning_rate=0.001,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,2.787500,2.195174,3.736400,16.989700
2,2.602500,2.122182,3.946000,16.960000


/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1178: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Checkpoint destination directory trained/checkpoint-2000 already exists and is non-empty. Saving will proceed but saved results may be invalid.
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1178: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


TrainOutput(global_step=2078, training_loss=2.7669836710690303, metrics={'train_runtime': 892.68, 'train_samples_per_second': 37.238, 'train_steps_per_second': 2.328, 'total_flos': 3019997016806400.0, 'train_loss': 2.7669836710690303, 'epoch': 2.0})

In [ ]:
generate(prefix + "Как тебя зовут?")

In [ ]:
generate(prefix + "Как у тебя имя?")

**Задание 1.** Сделайте ручное тестирование качества перевода. Возьмите пять случайных фраз из валидационной (тестовой) выборки датасета. Сделайте вывод.

**Задание 2.** Продолжите обучение модели ещё на 3 эпохах (запустите повторно код на обучение без инициализации исходной модели, либо запустите обучение исходной модели, но на 10 эпохах). Проанализируйте результаты на выбранных фразах в первом задании и сделайте выводы.

**Задание 3.** Соберите свой датасет из песен и их переводов (набор песен и исполнительный выберите самостоятельно). Используйте инструменты `web crawling` (например, библиотеку `beautifulsoup`) и, например, <a href="https://www.amalgama-lab.com/songs/">сайт с переводами песен</a> для составления своего датасета. В качестве единицы датасета используйте `фразу`/`предложение`/`строфы` (выберите самостоятельно). Ориентируйтесь на размер датасета - 2000 пар фраз.

Датасет сохраните в `json` формате. Для загрзки датасета из `json` файла используйте:

```python
from datasets import load_dataset
dataset = load_dataset("json", data_files="my_file.json")
```

Дообучите оригинальную модель на собственном датасете и проанализируйте качество и стиль перевода. Сделайте выводы.